# Freight Cost Minimization Supply Chain Logistics Brunel - Operations Research Project

*Operations Research Project - Freight Cost Minimization Supply Chain Logistics Brunel* aims to provide a business solution for Minimizing Freight Carrier Transportation Costs by constructing **Optimization Model** using **Linear Programming (LP) Simplex Method** in Operations Research. The Mathematical Optimization Model determines the most cost-efficient carrier assignment by considering operational and transportation constraints. Deployed using **PuLP (CBC Solver)**, the Optimization Model integrates **Decision Variables**, **Objective Function**, and **Constraints** to obtain the optimal solution. The Model combines **Customer Orders, Carrier Availability , Freight Rates, Transportation Routes, and Service-Level** requirements to identify optimal shipment allocation strategy.

## IMPORTING MODULES

In [1]:
import warnings
warnings.filterwarnings("ignore") 

import os, time 
import numpy as np 
import pandas as pd

%matplotlib inline 
import matplotlib 
import matplotlib.pyplot as plt 
import matplotlib.gridspec as gridspec 
import matplotlib.patches as mpatches 
from matplotlib.ticker import FuncFormatter 

import pulp 
from tabulate import tabulate 

## LOADING DATA

In [2]:
data_file = "supply_chain.xlsx" 
assert os.path.exists(data_file), "Dataset Not Found: suuply_chain.xlsx" 

orders = pd.read_excel(data_file, sheet_name = "OrderList") 
freight = pd.read_excel(data_file, sheet_name = "FreightRates") 

freight.columns = freight.columns.str.strip() 
freight["mode_dsc"] = freight["mode_dsc"].str.strip() 
freight["svc_cd"] = freight["svc_cd"].str.strip() 
freight_clean = freight[freight["rate"] > 0].copy().reset_index(drop = True) 

print(f"OrderList    : {len(orders):,} orders") 
print(f"FreightRates : {len(freight_clean):,} valid rate rows | {freight_clean["Carrier"].nunique()} carriers") 
print() 

OrderList    : 9,215 orders
FreightRates : 1,540 valid rate rows | 9 carriers



##  EXPLORATORY DATA ANALYSIS

In [3]:
opt_orders = orders[orders["Service Level"] != "CRF"].copy().reset_index(drop = True) 

print(f"  Total Orders                      : {len(orders):,}") 
print(f"  CRF Orders (Excluded)             : {(orders["Service Level"] == "CRF").sum():,}") 
print(f"  Optimisable Orders (DTP + DTD)    : {len(opt_orders):,}") 
print(f"    DTP (Door-To-Port)              : {(opt_orders["Service Level"] == "DTP").sum():,}")
print(f"    DTD (Door-To-Door)              : {(opt_orders["Service Level"] == "DTD").sum():,}")
print(f"  Unique Origin Ports               : {(opt_orders["Origin Port"]).nunique()}") 
print(f"  Unique Destination Ports          : {(opt_orders["Destination Port"]).nunique()}") 
print()

w = opt_orders["Weight"] 

print(f"Min     : {w.min():.3f} kg    Max    : {w.max():,.2f} kg") 
print(f"Mean    : {w.mean():.2f} kg    Median : {w.median():.2f} kg") 
print(f"Std Dev : {w.std():.2f} kg") 
print()

print(f"  Carriers        : {(freight_clean["Carrier"]).nunique()}  ({", ".join(freight_clean["Carrier"].unique())})") 
print(f"  Transport Modes : {freight_clean["mode_dsc"].value_counts().to_dict()}") 
print(f"  Rate Range      : ${freight_clean["rate"].min():.4f} - ${freight_clean["rate"].max():.2f} per kg") 
print(f"  Mean Rate       : ${freight_clean["rate"].mean():.4f} per kg") 
print(f"  Median Rate     : ${freight_clean["rate"].median():.4f} per kg") 
print()

lanes = opt_orders[["Origin Port", "Destination Port", "Service Level"]].drop_duplicates() 
print(tabulate(lanes, headers = lanes.columns, tablefmt = "rounded_outline", showindex = False)) 
for _, lane in lanes.iterrows(): 
    cnt = len(freight_clean[ 
        (freight_clean["orig_port_cd"] == lane["Origin Port"]) & 
        (freight_clean["dest_port_cd"] == lane["Destination Port"]) & 
        (freight_clean["svc_cd"] == lane["Service Level"]) 
    ]["Carrier"].unique()) 
    print(f"  ({lane["Origin Port"]} -> {lane["Destination Port"]}, {lane["Service Level"]}): {cnt} carriers available") 

print()

  Total Orders                      : 9,215
  CRF Orders (Excluded)             : 854
  Optimisable Orders (DTP + DTD)    : 8,361
    DTP (Door-To-Port)              : 6,218
    DTD (Door-To-Door)              : 2,143
  Unique Origin Ports               : 2
  Unique Destination Ports          : 1

Min     : 0.000 kg    Max    : 2,338.41 kg
Mean    : 17.70 kg    Median : 4.40 kg
Std Dev : 59.78 kg

  Carriers        : 9  (V444_6, V444_8, V444_9, V444_2, V444_1, V444_0, V444_5, V444_4, V444_7)
  Transport Modes : {'AIR': 1511, 'GROUND': 29}
  Rate Range      : $0.0332 - $128.03 per kg
  Mean Rate       : $2.8927 per kg
  Median Rate     : $1.6612 per kg

╭───────────────┬────────────────────┬─────────────────╮
│ Origin Port   │ Destination Port   │ Service Level   │
├───────────────┼────────────────────┼─────────────────┤
│ PORT09        │ PORT09             │ DTP             │
│ PORT04        │ PORT09             │ DTP             │
│ PORT04        │ PORT09             │ DTD            

## DATA PREPARATION

In [4]:
nSample = 500 
np.random.seed(42) 

sample_idx    = np.random.choice(len(opt_orders), size = nSample, replace = False) 
sample_orders = opt_orders.iloc[sample_idx].reset_index(drop = True) 

print(f"Random Sample                       : {nSample} Orders") 
print(f"Service Mix In Sample               : {sample_orders["Service Level"].value_counts().to_dict()}") 
print()

order_options = [] 
for i, row in sample_orders.iterrows(): 
    original, destination, service, weight = (row["Origin Port"], row["Destination Port"], row["Service Level"], row["Weight"]) 

    filtering = (
        (freight_clean["orig_port_cd"] == original) &
        (freight_clean["dest_port_cd"] == destination) &
        (freight_clean["svc_cd"] == service) &
        (freight_clean["minm_wgh_qty"] <= weight) &
        (freight_clean["max_wgh_qty"] >= weight)
    )
    candidates = freight_clean[filtering].copy() 

    if candidates.empty: 
        filtering2 = (
            (freight_clean["orig_port_cd"] == original) &
            (freight_clean["dest_port_cd"] == destination) &
            (freight_clean["svc_cd"] == service)
        ) 
        candidates = freight_clean[filtering2].copy()

    if not candidates.empty: 
        candidates["total_cost"] = candidates.apply(lambda r: max(r["minimum cost"], r["rate"] * weight), axis = 1) 
        for _, c in candidates.iterrows(): 
            order_options.append({ 
                "order_idx"  : i,
                "carrier"    : c["Carrier"],
                "cost"       : c["total_cost"],
                "tpt_days"   : c["tpt_day_cnt"],
                "mode"       : c["mode_dsc"],
                "rate_per_kg": c["rate"]
            })

options = pd.DataFrame(order_options) 
print(f"Total OrderxCarrier Options         : {len(options):,}") 
print(f"Orders With > 2 Carrier Option      : {(options["order_idx"].value_counts() > 2).sum()}") 
print(f"Orders With Missing Carrier Options : {nSample - options["order_idx"].nunique()}") 
print(f"Average Carrier Options Per Order   : {len(options)/options["order_idx"].nunique():.2f}") 

Random Sample                       : 500 Orders
Service Mix In Sample               : {'DTP': 360, 'DTD': 140}

Total OrderxCarrier Options         : 1,924
Orders With > 2 Carrier Option      : 140
Orders With Missing Carrier Options : 0
Average Carrier Options Per Order   : 3.85


## LINEAR PROGRAMMING MODEL FORMULATION

In [5]:
lp = pulp.LpProblem("Freight_Cost_Minimization", pulp.LpMinimize) 

x = {} 
for _, opt in options.iterrows(): 
    key = (opt["order_idx"], opt["carrier"]) 
    if key not in x: 
        x[key] = pulp.LpVariable(f"x_{int(opt["order_idx"])}_{opt["carrier"]}", lowBound = 0, upBound = 1, cat = "Continuous") 

lp += pulp.lpSum(opt["cost"] * x[(opt["order_idx"], opt["carrier"])] for _, opt in options.iterrows() if (opt["order_idx"], opt["carrier"]) in x), "Total_Freight_Cost" 

orders_in_lp = options["order_idx"].unique() 
for i in orders_in_lp: 
    carriers_i = options[options["order_idx"] == i]["carrier"].unique() 
    lp += pulp.lpSum(x[(i, c)] for c in carriers_i if (i, c) in x) == 1, f"Assign_Order_{int(i)}" 

print(f"Linear Programming Variables   : {len(x)}") 
print(f"Linear Programming Constraints : {len(lp.constraints)}") 

Linear Programming Variables   : 839
Linear Programming Constraints : 500


## SOLVING LINEAR PROGRAMMING

In [6]:
t0 = time.time() 
status = lp.solve(pulp.PULP_CBC_CMD(msg = 0)) 
t_solve = time.time() - t0 

optimal_cost = pulp.value(lp.objective) 

print(f"Status                                    : {pulp.LpStatus[status]}") 
print(f"Optimal Total Cost With 500 Sample Orders : ${optimal_cost:,.4f}")

Status                                    : Optimal
Optimal Total Cost With 500 Sample Orders : $2,917.8564


## RESULTS AND ANALYSIS

In [7]:
assign_result = []
for _, opt in options.iterrows():
    key = (opt["order_idx"], opt["carrier"])
    if key in x and pulp.value(x[key]) is not None and pulp.value(x[key]) > 0.5:
        assign_result.append({
            "Order"    : int(opt["order_idx"]),
            "Carrier"  : opt["carrier"],
            "Cost_USD" : opt["cost"],
            "TPT_Days" : opt["tpt_days"],
            "Mode"     : opt["mode"],
            "Rate_Kg"  : opt["rate_per_kg"]
        })
assign_result_df = pd.DataFrame(assign_result).sort_values("Order").drop_duplicates(subset = "Order")

print(tabulate(assign_result_df.head(15), headers = ["Order", "Carrier", "Cost ($)", "Transit (days)", "Mode", "Rate/kg ($)"], tablefmt = "rounded_outline", floatfmt = ".4f", showindex = False))
print()

carrier_summary = (assign_result_df.groupby("Carrier").agg(Orders = ("Order", "count"), Total_Cost = ("Cost_USD", "sum"), Avg_Cost = ("Cost_USD", "mean"), Avg_TPT = ("TPT_Days", "mean")).sort_values("Orders", ascending = False).reset_index())

print(tabulate(carrier_summary, headers = ["Carrier", "Orders", "Total Cost ($)", "Avg Cost/Order ($)", "Avg Transit (days)"], tablefmt = "rounded_outline", floatfmt = ".4f", showindex = False))
print()

mode_summary = (assign_result_df.groupby("Mode").agg(Orders = ("Order", "count"), Total_Cost = ("Cost_USD", "sum")).sort_values("Orders", ascending = False).reset_index())
mode_summary["Share_%"] = mode_summary["Orders"]/len(assign_result_df) * 100

print(tabulate(mode_summary, headers = ["Mode", "Orders", "Total Cost ($)", "Share (%)"], tablefmt = "rounded_outline", floatfmt = ".2f", showindex = False))
print()

historical_cost = options.groupby("order_idx")["cost"].max().sum()
cheapest_cost = options.groupby("order_idx")["cost"].min().sum()
saving = historical_cost - optimal_cost
saving_pct = saving / historical_cost * 100 
scale = len(opt_orders)/nSample
fullDataOptimal = optimal_cost * scale
fullDataHistorical = historical_cost * scale
fullDataCheapest = cheapest_cost * scale
fullDataSaving = fullDataHistorical - fullDataOptimal 

fullSummary = [
    ["Historical (Most Expensive)", f"${historical_cost:,.4f}", f"${fullDataHistorical:,.0f}", f"Baseline"],
    ["Greedy (Most Cheap)", f"${cheapest_cost:,.4f}", f"${fullDataCheapest:,.0f}", f"${historical_cost - cheapest_cost:,.4f}"],
    ["Linear Programming Simplex Optimal", f"${optimal_cost:,.4f}", f"${fullDataOptimal:,.0f}", f"${saving:,.4f} ({saving_pct:.1f}%)"]
]

print(tabulate(fullSummary, headers = ["Optimal Strategy", "Cost (500 Orders)", "Cost (8,361 Orders)", "Savings To Historical"], tablefmt = "rounded_outline"))
print()

dual_result = []
count = 0
for name, constraint in lp.constraints.items():
    if count >= 10:
        break
    if constraint.pi is not None:
        dp = constraint.pi
    else:
        dp = 0.0

    dual_result.append([f"Order {count+1}", f"${dp:.4f}"])
    count += 1

print(tabulate(dual_result, headers = ["Order", "Dual Price ($)"], tablefmt = "rounded_outline"))
print()

╭─────────┬───────────┬────────────┬──────────────────┬────────┬───────────────╮
│   Order │ Carrier   │   Cost ($) │   Transit (days) │ Mode   │   Rate/kg ($) │
├─────────┼───────────┼────────────┼──────────────────┼────────┼───────────────┤
│       0 │ V444_0    │     1.4992 │                3 │ AIR    │        0.0484 │
│       1 │ V444_0    │     3.4552 │                3 │ AIR    │        0.0824 │
│       2 │ V444_0    │     2.9263 │                3 │ AIR    │        0.0484 │
│       3 │ V444_0    │     3.4552 │                3 │ AIR    │        0.0824 │
│       4 │ V444_0    │     1.4992 │                3 │ AIR    │        0.0484 │
│       5 │ V444_0    │     1.4992 │                3 │ AIR    │        0.0484 │
│       6 │ V444_0    │     1.4992 │                3 │ AIR    │        0.0484 │
│       7 │ V444_0    │     3.4552 │                3 │ AIR    │        0.0824 │
│       8 │ V444_0    │     2.4921 │                3 │ AIR    │        0.0484 │
│       9 │ V444_0    │     

In [8]:
fullSummary_df = pd.DataFrame({
    "Optimal Strategy"            : ["Historical (Most Expensive)", "Greedy (Most Cheap)", "Linear Programming Simplex Optimal"],
    "Cost (500 Orders) ($)"       : [historical_cost, cheapest_cost, optimal_cost],
    "Cost (8,361 Orders) ($)"     : [fullDataHistorical, fullDataCheapest, fullDataOptimal],
    "Savings To Historical ($)"   : [None, historical_cost - cheapest_cost, saving]
})

dual_result = []
count = 0
for name, constraint in lp.constraints.items():
    if constraint.pi is not None:
        dp = constraint.pi
    else:
        dp = 0.0

    dual_result.append({"Order" : count+1, "Dual_Price_USD": dp})
    count += 1
dual_result_df = pd.DataFrame(dual_result).sort_values("Order").reset_index(drop = True)

assign_result_df.to_csv("optimal_assignment.csv", index = False)
carrier_summary.to_csv("carrier_summary.csv", index = False)
mode_summary.to_csv("mode_summary.csv", index = False)
fullSummary_df.to_csv("optimal_strategy.csv", index = False)
dual_result_df.to_csv("dual_price.csv", index = False)

print("Saved: Result and Analysis CSV Files")

Saved: Result and Analysis CSV Files
